In [1]:
import os, shutil, random
from pathlib import Path

In [2]:
src = Path("/home/somethink/parking_system/yolo_plate_dataset")
dst = Path("/home/somethink/parking_system/yolo_plate_data")
val_ratio = 0.2
random_seed = 42

In [9]:
random.seed(random_seed)

images = sorted([
    f for f in src.glob("*.jpg")
    if (src / f.with_suffix(".txt").name).exists()
])
print(f"Tong so anh co label: {len(images)}")

random.shuffle(images)
n_val = int(len(images) * val_ratio)
splits = {"val": images[:n_val], "train": images[n_val:]}

for split, imgs in splits.items():
    img_dir = dst / "images" / split
    lbl_dir = dst / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in imgs:
        shutil.copy(img, img_dir / img.name)
        shutil.copy(src / img.with_suffix(".txt").name,
                    lbl_dir / img.with_suffix(".txt").name)
    print(f"  {split}: {len(imgs)} anh")

yaml_content = (
    f"path: {dst.resolve()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 1\n"
    "names: ['plate']\n"
)
Path("plates.yaml").write_text(yaml_content)
print("\nTao file: plates.yaml")
print("Chay train:")
print("  yolo train model=yolov8n.pt data=plates.yaml epochs=100 imgsz=640 batch=16 device=0")



Tong so anh co label: 8259
  val: 1651 anh
  train: 6608 anh

Tao file: plates.yaml
Chay train:
  yolo train model=yolov8n.pt data=plates.yaml epochs=100 imgsz=640 batch=16 device=0


In [6]:
!yolo train model=yolov8n.pt data=plates.yaml epochs=35 imgsz=640 batch=4 amp=True device=0 workers=2

New https://pypi.org/project/ultralytics/8.4.21 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plates.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=35, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train5, nbs=64, nms=False, opset=None, optimize

In [7]:
!MPLBACKEND=Agg yolo detect val model=/home/somethink/parking_system/runs/detect/train5/weights/best.pt data=plates.yaml device=0 plots=False

Ultralytics 8.4.19 🚀 Python-3.10.12 torch-2.8.0 CUDA:0 (Orin, 7620MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 482.4±140.0 MB/s, size: 42.2 KB)
val: Scanning /home/somethink/parking_system/yolo_plate_data/labels/val.cache... 1651 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1651/1651 64.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 104/104 3.7it/s 28.4s0.3s
                   all       1651       1686      0.994      0.991      0.995      0.746
Speed: 0.9ms preprocess, 10.6ms inference, 0.0ms loss, 2.0ms postprocess per image
💡 Learn more at https://docs.ultralytics.com/modes/val
